# 🔄 AuraNet Development Template

**Universal notebook that works in both VS Code and Google Colab**

---

In [ ]:
# =============================================================================
# ENVIRONMENT DETECTION & SETUP
# =============================================================================

import os
import sys

# Detect environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_VSCODE = not IN_COLAB

print(f"🖥️  Environment: {'Google Colab' if IN_COLAB else 'VS Code / Local'}")

# Configure paths based on environment
if IN_COLAB:
    # === COLAB SETUP ===
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    
    # Clone or update repository
    REPO_URL = "https://github.com/YOUR_USERNAME/auranet.git"  # ← UPDATE THIS
    PROJECT_ROOT = "/content/auranet"
    
    if not os.path.exists(PROJECT_ROOT):
        print("📥 Cloning repository...")
        !git clone {REPO_URL} {PROJECT_ROOT}
    else:
        print("🔄 Pulling latest changes...")
        !cd {PROJECT_ROOT} && git pull
    
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, PROJECT_ROOT)
    
    # Google Drive paths for persistent storage
    DRIVE_ROOT = "/content/drive/MyDrive/AuraNet"
    DATA_DIR = f"{DRIVE_ROOT}/datasets"
    CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"
    OUTPUT_DIR = f"{DRIVE_ROOT}/outputs"
    
    # Create directories on Drive
    for d in [DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
        os.makedirs(d, exist_ok=True)
    
    # Install dependencies
    print("📦 Installing dependencies...")
    !pip install -q torch torchaudio librosa soundfile tqdm pyyaml
    
else:
    # === LOCAL / VS CODE SETUP ===
    PROJECT_ROOT = "/Users/vinoth-14902/auranet"
    sys.path.insert(0, PROJECT_ROOT)
    
    DATA_DIR = os.path.join(PROJECT_ROOT, "datasets")
    CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
    OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")
    
    for d in [DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
        os.makedirs(d, exist_ok=True)

print(f"\n📂 Project Root: {PROJECT_ROOT}")
print(f"📊 Data Dir: {DATA_DIR}")
print(f"💾 Checkpoint Dir: {CHECKPOINT_DIR}")

🖥️  Environment: VS Code / Local

📂 Project Root: /content
📊 Data Dir: /content/datasets
💾 Checkpoint Dir: /content/checkpoints


In [ ]:
# =============================================================================
# GPU / DEVICE SETUP
# =============================================================================

import torch

def get_device():
    """Get the best available device."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"🎮 GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device("mps")
        print("🍎 Using Apple Silicon GPU (MPS)")
    else:
        device = torch.device("cpu")
        print("⚠️  Using CPU - training will be slow!")
        if IN_COLAB:
            print("💡 Enable GPU: Runtime → Change runtime type → GPU")
    return device

DEVICE = get_device()
print(f"\n✅ Device: {DEVICE}")

In [ ]:
# =============================================================================
# IMPORTS & CONFIGURATION
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import yaml

# Import AuraNet modules
try:
    from model import AuraNet, create_auranet
    from dataset import AuraNetDataset
    from loss import AuraNetLoss
    from utils.stft import CausalSTFT
    print("✅ AuraNet modules imported")
except ImportError as e:
    print(f"⚠️  Import error: {e}")
    print("   Make sure you're in the project directory")

# Load config
config_path = os.path.join(PROJECT_ROOT, "config.yaml")
if os.path.exists(config_path):
    with open(config_path) as f:
        CONFIG = yaml.safe_load(f)
    print("✅ Config loaded")
else:
    CONFIG = {}
    print("⚠️  No config.yaml found, using defaults")

---

## 🏋️ Training Section

The cells below are for training. Run locally with `--debug` for quick tests, or in Colab for full GPU training.

In [ ]:
# =============================================================================
# TRAINING CONFIGURATION
# =============================================================================

# Override settings based on environment
if IN_COLAB:
    # Full training on Colab
    BATCH_SIZE = 16
    NUM_EPOCHS = 100
    USE_SYNTHETIC = False  # Use real data from Drive
else:
    # Quick debug on local
    BATCH_SIZE = 4
    NUM_EPOCHS = 2
    USE_SYNTHETIC = True  # Use synthetic data for testing

print(f"\n📋 Training Config:")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Data: {'Synthetic' if USE_SYNTHETIC else 'Real'}")

In [ ]:
# =============================================================================
# CREATE MODEL & DATASET
# =============================================================================

# Create model
model = create_auranet(CONFIG)
model = model.to(DEVICE)
print(f"\n🧠 Model created: {model.count_parameters():,} parameters")

# Create dataset
dataset = AuraNetDataset(
    clean_dirs=[os.path.join(DATA_DIR, "speech"), os.path.join(DATA_DIR, "music")],
    noise_dir=os.path.join(DATA_DIR, "noise"),
    synthetic_mode=USE_SYNTHETIC,
    num_synthetic=100 if USE_SYNTHETIC else 0,
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2 if IN_COLAB else 0,
    pin_memory=True,
)

print(f"📊 Dataset: {len(dataset)} samples, {len(dataloader)} batches")

In [ ]:
# =============================================================================
# TRAINING LOOP
# =============================================================================

# Setup
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
criterion = AuraNetLoss()
stft = CausalSTFT().to(DEVICE)

# Load checkpoint if exists
start_epoch = 0
best_loss = float('inf')
checkpoint_path = os.path.join(CHECKPOINT_DIR, "latest.pt")

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint.get('best_loss', float('inf'))
    print(f"✅ Resumed from epoch {start_epoch}")

# Training loop
print(f"\n🚀 Starting training from epoch {start_epoch + 1}...")

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in pbar:
        noisy_stft = batch['noisy_stft'].to(DEVICE)
        clean_stft = batch['clean_stft'].to(DEVICE)
        clean_audio = batch['clean_audio'].to(DEVICE)
        
        optimizer.zero_grad()
        
        enhanced_stft, _, _ = model(noisy_stft)
        enhanced_audio = stft.inverse(enhanced_stft)
        
        loss, _ = criterion(enhanced_stft, clean_stft, enhanced_audio, clean_audio)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")
    
    # Save checkpoint
    is_best = avg_loss < best_loss
    if is_best:
        best_loss = avg_loss
    
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_loss': best_loss,
    }, checkpoint_path)
    
    if is_best:
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_model.pt"))
        print(f"💾 Best model saved!")

print(f"\n✅ Training complete! Best loss: {best_loss:.4f}")

---

## 🎵 Inference Section

In [ ]:
# =============================================================================
# INFERENCE
# =============================================================================

import torchaudio
from IPython.display import Audio, display

# Load best model
best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    print("✅ Loaded best model")
else:
    print("⚠️  No saved model found, using current weights")

model.eval()

@torch.no_grad()
def enhance_audio(audio_path_or_tensor, sample_rate=16000):
    """Enhance audio using AuraNet."""
    
    # Load if path
    if isinstance(audio_path_or_tensor, str):
        audio, sr = torchaudio.load(audio_path_or_tensor)
        if sr != sample_rate:
            audio = torchaudio.transforms.Resample(sr, sample_rate)(audio)
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)
    else:
        audio = audio_path_or_tensor
        if audio.dim() == 1:
            audio = audio.unsqueeze(0)
    
    audio = audio.to(DEVICE)
    
    # Process
    noisy_stft = stft(audio)
    enhanced_stft, _, _ = model(noisy_stft)
    enhanced_audio = stft.inverse(enhanced_stft)
    
    return enhanced_audio.cpu().squeeze()

print("✅ Inference function ready")

In [ ]:
# =============================================================================
# TEST WITH SYNTHETIC AUDIO
# =============================================================================

# Create noisy test audio
sample_rate = 16000
duration = 2.0
t = torch.linspace(0, duration, int(sample_rate * duration))

# Clean signal: harmonics
clean = sum((1/h) * torch.sin(2 * 3.14159 * 150 * h * t) for h in range(1, 6))
clean = clean / clean.abs().max() * 0.5

# Add noise
noise = torch.randn_like(t) * 0.15
noisy = clean + noise

# Enhance
enhanced = enhance_audio(noisy, sample_rate)

# Display
print("🔊 Noisy Audio:")
display(Audio(noisy.numpy(), rate=sample_rate))

print("🔊 Enhanced Audio:")
display(Audio(enhanced.numpy(), rate=sample_rate))

print("🔊 Clean Reference:")
display(Audio(clean.numpy(), rate=sample_rate))

In [ ]:
# =============================================================================
# PROCESS YOUR OWN FILE (Colab only)
# =============================================================================

if IN_COLAB:
    from google.colab import files
    
    print("📤 Upload your audio file:")
    uploaded = files.upload()
    
    if uploaded:
        filename = list(uploaded.keys())[0]
        print(f"\n🔄 Processing: {filename}")
        
        # Load and enhance
        enhanced = enhance_audio(filename)
        
        # Save
        output_path = f"enhanced_{filename}"
        torchaudio.save(output_path, enhanced.unsqueeze(0), 16000)
        
        # Play
        print("\n🔊 Enhanced audio:")
        display(Audio(enhanced.numpy(), rate=16000))
        
        # Download
        files.download(output_path)
else:
    print("ℹ️  File upload only available in Colab")
    print("   Locally, use: enhanced = enhance_audio('path/to/file.wav')")

---

## 📤 Save & Sync

After making changes, sync back to GitHub:

In [ ]:
# =============================================================================
# GIT SYNC (Colab only)
# =============================================================================

if IN_COLAB:
    # Configure git (first time only)
    !git config --global user.email "your@email.com"
    !git config --global user.name "Your Name"
    
    # Check status
    !git status
    
    # Uncomment to push changes:
    # !git add -A
    # !git commit -m "Training updates from Colab"
    # !git push
else:
    print("ℹ️  Use VS Code terminal for git commands locally")